In [ ]:
!pip install openai supabase langchain langchain-community langchain-openai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.0/730.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 11.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently

In [ ]:
from google.colab import files
uploaded = files.upload()  # This will open a file picker — select your SBC PDF

Saving SBC 201 - The Saudi General Building Code.pdf to SBC 201 - The Saudi General Building Code.pdf


In [ ]:
pdf_filename = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_filename}")

Uploaded: SBC 201 - The Saudi General Building Code.pdf


In [ ]:
# @title
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from supabase import create_client

# === FILL IN YOUR CREDENTIALS ===
SUPABASE_URL = "--"
SUPABASE_KEY = "--"
OPENAI_API_KEY = "--"

# === INIT ===
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

# === LOAD & SPLIT PDF ===
print(f"Loading PDF: {pdf_filename}")
loader = PyPDFLoader(pdf_filename)
pages = loader.load()
print(f"Loaded {len(pages)} pages")

# Clean null characters from text
for page in pages:
    page.page_content = page.page_content.replace("\u0000", "")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")

# === EMBED & UPLOAD ===
batch_size = 50
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    texts = [chunk.page_content.replace("\u0000", "") for chunk in batch]
    metadatas = [{"source": chunk.metadata.get("source", "SBC"),
                  "page": chunk.metadata.get("page", 0)} for chunk in batch]

    print(f"Embedding batch {i // batch_size + 1}/{(len(chunks) - 1) // batch_size + 1}...")
    vectors = embeddings.embed_documents(texts)

    rows = []
    for text, metadata, vector in zip(texts, metadatas, vectors):
        rows.append({
            "content": text,
            "metadata": metadata,
            "embedding": vector
        })

    supabase.table("sbc_documents").insert(rows).execute()
    print(f"  Uploaded {len(rows)} rows")

print(f"\nDone! Total chunks uploaded: {len(chunks)}")

Loading PDF: SBC 201 - The Saudi General Building Code.pdf
Loaded 2200 pages
Split into 10313 chunks
Embedding batch 1/207...
  Uploaded 50 rows
Embedding batch 2/207...
  Uploaded 50 rows
Embedding batch 3/207...
  Uploaded 50 rows
Embedding batch 4/207...
  Uploaded 50 rows
Embedding batch 5/207...
  Uploaded 50 rows
Embedding batch 6/207...
  Uploaded 50 rows
Embedding batch 7/207...
  Uploaded 50 rows
Embedding batch 8/207...
  Uploaded 50 rows
Embedding batch 9/207...
  Uploaded 50 rows
Embedding batch 10/207...
  Uploaded 50 rows
Embedding batch 11/207...
  Uploaded 50 rows
Embedding batch 12/207...
  Uploaded 50 rows
Embedding batch 13/207...
  Uploaded 50 rows
Embedding batch 14/207...
  Uploaded 50 rows
Embedding batch 15/207...
  Uploaded 50 rows
Embedding batch 16/207...
  Uploaded 50 rows
Embedding batch 17/207...
  Uploaded 50 rows
Embedding batch 18/207...
  Uploaded 50 rows
Embedding batch 19/207...
  Uploaded 50 rows
Embedding batch 20/207...
  Uploaded 50 rows
Embeddin

In [ ]:
# Delete in batches
while True:
    result = supabase.table("sbc_documents").select("id").limit(500).execute()
    if not result.data:
        break
    ids = [row["id"] for row in result.data]
    supabase.table("sbc_documents").delete().in_("id", ids).execute()
    print(f"Deleted {len(ids)} rows...")

print("All old chunks cleared!")

Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 500 rows...
Deleted 126 rows...
All old chunks cleared!


In [ ]:
# LARGER chunks with more overlap
splitter2 = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400,
    separators=["\n\n\n", "\n\n", "\n", ". ", " ", ""]
)
chunks2 = splitter2.split_documents(pages)
print(f"Split into {len(chunks2)} chunks")

# Upload
batch_size = 50
failed_batches = []

for i in range(0, len(chunks2), batch_size):
    batch = chunks2[i:i + batch_size]
    texts = [chunk.page_content.replace("\u0000", "") for chunk in batch]
    metadatas = [{
        "source": "SBC 201 - Saudi General Building Code 2024",
        "page": chunk.metadata.get("page", 0),
        "chunk_index": i + j
    } for j, chunk in enumerate(batch)]

    print(f"Embedding batch {i // batch_size + 1}/{(len(chunks2) - 1) // batch_size + 1}...")

    try:
        vectors = embeddings.embed_documents(texts)
        rows = []
        for text, metadata, vector in zip(texts, metadatas, vectors):
            rows.append({
                "content": text,
                "metadata": metadata,
                "embedding": vector
            })
        supabase.table("sbc_documents").insert(rows).execute()
        print(f"  Uploaded {len(rows)} rows")
    except Exception as e:
        print(f"  FAILED batch {i // batch_size + 1}: {e}")
        failed_batches.append(i)

print(f"\nDone! Total chunks: {len(chunks2)}")
if failed_batches:
    print(f"Failed batches: {failed_batches}")

Split into 5354 chunks
Embedding batch 1/108...
  Uploaded 50 rows
Embedding batch 2/108...
  Uploaded 50 rows
Embedding batch 3/108...
  Uploaded 50 rows
Embedding batch 4/108...
  Uploaded 50 rows
Embedding batch 5/108...
  Uploaded 50 rows
Embedding batch 6/108...
  Uploaded 50 rows
Embedding batch 7/108...
  Uploaded 50 rows
Embedding batch 8/108...
  Uploaded 50 rows
Embedding batch 9/108...
  Uploaded 50 rows
Embedding batch 10/108...
  Uploaded 50 rows
Embedding batch 11/108...
  Uploaded 50 rows
Embedding batch 12/108...
  Uploaded 50 rows
Embedding batch 13/108...
  Uploaded 50 rows
Embedding batch 14/108...
  Uploaded 50 rows
Embedding batch 15/108...
  Uploaded 50 rows
Embedding batch 16/108...
  Uploaded 50 rows
Embedding batch 17/108...
  Uploaded 50 rows
Embedding batch 18/108...
  Uploaded 50 rows
Embedding batch 19/108...
  Uploaded 50 rows
Embedding batch 20/108...
  Uploaded 50 rows
Embedding batch 21/108...
  FAILED batch 21: {'message': 'JSON could not be generated'

In [ ]:
# Re-upload failed batches
for start in [1000, 1050]:
    batch = chunks2[start:start + 50]
    texts = [chunk.page_content.replace("\u0000", "") for chunk in batch]
    metadatas = [{
        "source": "SBC 201 - Saudi General Building Code 2024",
        "page": chunk.metadata.get("page", 0),
        "chunk_index": start + j
    } for j, chunk in enumerate(batch)]

    print(f"Retrying batch starting at {start}...")
    try:
        vectors = embeddings.embed_documents(texts)
        rows = []
        for text, metadata, vector in zip(texts, metadatas, vectors):
            rows.append({
                "content": text,
                "metadata": metadata,
                "embedding": vector
            })
        supabase.table("sbc_documents").insert(rows).execute()
        print(f"  Uploaded {len(rows)} rows")
    except Exception as e:
        print(f"  FAILED again: {e}")

print("Done!")

Retrying batch starting at 1000...
  Uploaded 50 rows
Retrying batch starting at 1050...
  Uploaded 50 rows
Done!


In [ ]:
# Search for "stair width" in the content
result = supabase.table("sbc_documents").select("id, content, metadata").ilike("content", "%stair width%").limit(5).execute()
print(f"Found {len(result.data)} chunks with 'stair width'")
for row in result.data:
    print(f"\n--- Page {row['metadata'].get('page')} ---")
    print(row['content'][:500])

Found 5 chunks with 'stair width'

--- Page 1037 ---
CHAPTER 10—MEANS OF EGRESS 
 
  
SBC 201-CC-2024 1010 
 
 For the first example, t he capacity criteria are 
more restrictive and, therefore, the minimum 
required width for each stairway is 1334 mm.  
2 Determine the minimum required stairway 
width with a se cond-floor occupant load of 
90: 
• 90 occupants multiplied by  7.62 mm = 686 
mm minimum. 
• 686 mm divided by two stairways is 343 mm 
or  
• Section 1011.2 prescribes that the width of 
an interior stairway cannot be less than 1118 
mm.

--- Page 1105 ---
CHAPTER 10—MEANS OF EGRESS 
 
  
SBC 201-CC-2024 1078 
 
 Exception 1, allowing for a clear headroom of 
2000 mm for spiral stairs, correlates with the 
provisions of Section 1011.10 .  
 Exception 2 recognizes a common method of 
stair-well construction in which the stringer on 
the open side of a stair is supported by the 
same floor joists or wall that supports the edge 
of the opening through which the stairway 
passes 